# Wav2Vec 2.0

In [2]:
import os
import pandas as pd
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer
)
import evaluate

# =========================
# 1. Setup Data
# =========================

DATA_PATH = "./data/CREMA-D/"
EMOTIONS = {'ANG': 0, 'DIS': 1, 'FEA': 2, 'HAP': 3, 'NEU': 4, 'SAD': 5}

data = []

for filename in os.listdir(DATA_PATH):
    if filename.endswith(".wav"):
        parts = filename.split('_')
        speaker_id = parts[0]
        emotion_str = parts[2]

        if emotion_str in EMOTIONS:
            data.append({
                "path": os.path.join(DATA_PATH, filename),
                "speaker": speaker_id,
                "label": EMOTIONS[emotion_str]
            })

df = pd.DataFrame(data)

# =========================
# 2. Speaker split
# =========================

unique_speakers = df['speaker'].unique().tolist()

train_speakers, test_speakers = train_test_split(
    unique_speakers,
    test_size=0.2,
    random_state=42
)

train_df = df[df['speaker'].isin(train_speakers)].reset_index(drop=True)
test_df = df[df['speaker'].isin(test_speakers)].reset_index(drop=True)

print(f"Total files: {len(df)}")
print(f"Training files: {len(train_df)}")
print(f"Testing files: {len(test_df)}")

# =========================
# 3. Convert to HF Dataset
# =========================

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

# =========================
# 4. Feature extractor
# =========================

model_id = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

MAX_DURATION = 3.0
MAX_LENGTH = int(MAX_DURATION * feature_extractor.sampling_rate)

# =========================
# 5. Preprocessing (SAFE)
# =========================

def preprocess_function(examples):
    audio_arrays = []

    for path in examples["path"]:
        array, _ = librosa.load(path, sr=16000)
        audio_arrays.append(array)

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = examples["label"]  # IMPORTANT

    return inputs

encoded_train = train_dataset.map(
    preprocess_function,
    remove_columns=["path", "speaker"],
    batched=True,
    batch_size=4
)

encoded_test = test_dataset.map(
    preprocess_function,
    remove_columns=["path", "speaker"],
    batched=True,
    batch_size=4
)

# =========================
# 6. Metrics
# =========================

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy_metric.compute(
        predictions=predictions,
        references=eval_pred.label_ids
    )

# =========================
# 7. Model
# =========================

num_labels = len(EMOTIONS)

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    id2label={str(v): k for k, v in EMOTIONS.items()},
    label2id={k: v for k, v in EMOTIONS.items()}
)

# =========================
# 8. Training config
# =========================

training_args = TrainingArguments(
    output_dir="./wav2vec2-crema-d-emotion",
    eval_strategy="epoch",   # ← NEW name in newer versions
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# =========================
# 9. Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    compute_metrics=compute_metrics,
)

# =========================
# 10. Train
# =========================

trainer.train()

Total files: 7442
Training files: 5890
Testing files: 1552


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 84349.80it/s]
Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprec

Epoch,Training Loss,Validation Loss,Accuracy
1,1.207789,1.128242,0.577320
2,0.786422,0.945892,0.669459
3,0.565739,0.971999,0.694588
4,0.398225,0.968933,0.714562
5,0.279768,1.102097,0.713918


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.50it/s]


TrainOutput(global_step=3685, training_loss=0.7612288037181062, metrics={'train_runtime': 747.2909, 'train_samples_per_second': 39.409, 'train_steps_per_second': 4.931, 'total_flos': 8.021059128288e+17, 'train_loss': 0.7612288037181062, 'epoch': 5.0})

In [4]:
import numpy as np

# Run predictions on the test set
predictions = trainer.predict(encoded_test)

# Get predicted labels
preds = np.argmax(predictions.predictions, axis=1)

# True labels
labels = predictions.label_ids

# Compute accuracy manually
test_accuracy = (preds == labels).mean()
print("✅ Test Accuracy:", test_accuracy)

# If you want the built-in metrics returned by compute_metrics:
print("Full metrics:", predictions.metrics)

✅ Test Accuracy: 0.7145618556701031
Full metrics: {'test_loss': 0.9689326882362366, 'test_accuracy': 0.7145618556701031, 'test_runtime': 21.605, 'test_samples_per_second': 71.835, 'test_steps_per_second': 8.979}
